# 2. Data analysis and band calculations

Compute Hamiltonian errors and ensemble disagreement with `tools/error_analysis.py`, and calculate DFT-reference and ensemble-mean bands with the existing `tools/sparse_calc.py`. The included inputs are real archived calculations; see [data provenance](data/README.md). This notebook runs on a CPU without retraining models or performing new DFT calculations. Run it before the visualization notebook.

In [1]:
from pathlib import Path
import configparser
import importlib.util
import json
import os
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'tools/error_analysis.py').is_file() and (p / 'models/config').is_dir())
DOC = ROOT / 'doc'
OUT = DOC / 'output'
OUT.mkdir(exist_ok=True)
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

In [2]:
import numpy as np
import h5py
analysis = load_module('ensemble_error_analysis', ROOT/'tools/error_analysis.py')

## Calculate errors from stored Hamiltonians

`analyze_errors` applies the stored mask to prediction-minus-DFT differences. `Avg_STD` is obtained from the matching per-structure `hamiltonians_std.h5` files. The sample retains the numerical arrays from the archive, but omits unrelated graph fields that this analysis does not read. It is intended to verify the workflow, not estimate performance from three structures.

In [3]:
sample = DOC / 'data/strain_example/test_result.h5'
results = analysis.analyze_errors(str(sample),verbose=False)
assert results['summary']['num_structures'] == 3
frame = pd.DataFrame(results['structures'])
assert frame['avg_std'].notna().all()
display(frame[['name','mse','mae','avg_std']].rename(
    columns={'mse':'MSE (eV^2)','mae':'MAE (eV)','avg_std':'mean nonzero STD (eV)'}))
analysis.save_to_csv(results,OUT/'sample_errors.csv')
analysis.save_to_json(results,OUT/'sample_errors.json')

,name,MSE (eV^2),MAE (eV),mean nonzero STD (eV)
0,t-0-0,0.000546,0.002422,0.001869
1,t-4-0,0.000066,0.001182,0.000836
2,t-8-1,0.000070,0.001175,0.000824


[+] 结果已保存至: <repository>\doc\output\sample_errors.csv
[+] 结果已保存至: <repository>\doc\output\sample_errors.json


## Verify against the archived analysis

The stored CSV contains summary rows and a non-English first-column header. The following adaptation changes that header only; numerical values retain their original units. These checks compare the re-evaluated HDF5 errors and mean standard deviations with the independently archived table.

In [4]:
def read_archived_table(path):
    import csv
    with path.open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        rows = [row for row in reader if row and row[0].startswith('t-')]
    assert rows and all(len(row) == len(header) for row in rows)
    frame = pd.DataFrame(rows, columns=['Structure Name', *header[1:]])
    for column in ('MSE','MAE','Avg_STD','Delta_cc','Delta_vdw','Delta_strain'):
        frame[column] = pd.to_numeric(frame[column],errors='coerce')
    return frame[frame['Structure Name'].astype(str).str.startswith('t-')].copy()

archive = read_archived_table(DOC/'data/bilayer_strain_error_analyze.csv').set_index('Structure Name')
for item in results['structures']:
    for computed,column in [('mse','MSE'),('mae','MAE'),('avg_std','Avg_STD')]:
        np.testing.assert_allclose(item[computed],archive.loc[item['name'],column],rtol=2e-6,atol=1e-10)
print('Verified MAE, MSE and Avg_STD against the archived table for all three structures.')

Verified MAE, MSE and Avg_STD against the archived table for all three structures.


## Calculate DFT-reference and ensemble-mean bands

The example includes matching real-space Hamiltonians, overlap matrices, reciprocal lattice, orbital types and atomic positions for the bilayer strain structure `t-4-0`. It uses the archived 16-band, 45-point Gamma-M-K-Gamma configuration, with the DFT Fermi energy in eV. The same settings and overlap matrices are used for both Hamiltonians.

The cell follows the input conventions of the existing `tools/band_calc.sh` workflow: the DFT Hamiltonian is copied under the solver's expected filename `hamiltonians_pred.h5`, while the ensemble mean already has that filename. Each calculation uses a fresh temporary working directory under `doc/output/bands` to avoid reusing a sparse-matrix cache from another Hamiltonian. The Python solver is an existing repository entry point; no replacement eigensolver is introduced here. To use other evaluated structures, supply their matching matrices and metadata and the appropriate band configuration.

In [5]:
import shutil
import subprocess
import tempfile

inputs = DOC/'data/band_example'
band_out = OUT/'bands'
band_out.mkdir(exist_ok=True)
metadata = json.loads((inputs/'info.json').read_text())
band_config = json.loads((inputs/'band.json').read_text())
assert band_config['fermi_level'] == metadata['fermi_level']
band_files = {}
for target, hamiltonian in [('dft','hamiltonians.h5'), ('mean','hamiltonians_pred.h5')]:
    with tempfile.TemporaryDirectory(prefix=target+'_', dir=band_out) as directory:
        work = Path(directory)
        for name in ('overlaps.h5','info.json','rlat.dat','orbital_types.dat','site_positions.dat'):
            shutil.copy2(inputs/name, work/name)
        shutil.copy2(inputs/hamiltonian, work/'hamiltonians_pred.h5')
        run = subprocess.run([sys.executable, str(ROOT/'tools/sparse_calc.py'),
                              '-i', str(work), '-o', str(work),
                              '--config', str(inputs/'band.json')],
                             cwd=ROOT, capture_output=True, text=True)
        (band_out/f'{target}_solver.log').write_text(run.stdout+'\n'+run.stderr, encoding='utf-8')
        if run.returncode:
            raise RuntimeError(f'{target} calculation failed; see doc/output/bands/{target}_solver.log')
        output = band_out/f'openmx_{target}.Band'
        shutil.copy2(work/'openmx.Band', output)
        band_files[target] = output
    print(f'Calculated {target}: {output.relative_to(DOC).as_posix()}')

Calculated dft: output/bands/openmx_dft.Band


Calculated mean: output/bands/openmx_mean.Band


## Check the band outputs

Use the repository's OpenMX parser to verify the expected band count, finite energies and identical k-point paths. The parser returns energies in eV relative to the Fermi energy. Plotting is kept in the next notebook.

In [6]:
bands = load_module('ensemble_band_plotter', ROOT/'tools/band_plotter.py')
parsed = {name: bands.parse_openmx_band(str(path)) for name,path in band_files.items()}
expected_points = sum(int(line.split()[0]) for line in band_config['k_data'])
for name, result in parsed.items():
    assert result[0] == band_config['num_band']
    assert result[5].shape == (band_config['num_band'], expected_points)
    assert np.isfinite(result[5]).all()
np.testing.assert_allclose(parsed['dft'][4], parsed['mean'][4], rtol=0, atol=0)
print(f'Validated DFT and ensemble-mean bands: {band_config["num_band"]} bands at {expected_points} k-points each.')

Validated DFT and ensemble-mean bands: 16 bands at 45 k-points each.
